# Hather: teacher prep run (rewrites + teacher comparison)

One committed run does three stages, so the model is downloaded once per teacher:

| # | Stage | Jobs file | Teacher | Output |
| --- | --- | --- | --- | --- |
| 1 | Rewrite real phone SMS into scams | `rewrite_jobs.jsonl` (800) | abliterated | `rewrite_messages.jsonl` |
| 2 | Teacher comparison | `runB_compare_jobs.jsonl` (100) | abliterated | `runB_compare_abliterated_labels.jsonl` |
| 3 | Teacher comparison | `runB_compare_jobs.jsonl` (100) | original | `runB_compare_original_labels.jsonl` |

Then `prep_summary.md` says which teacher agrees more with the true labels (use it for Run B).

**Before running:** Accelerator **GPU T4 x2**, Internet **on**, inputs: your jobs dataset (both jobs files) and
`llamacpp-t4`. *Save Version → Save & Run All*. In the Logs look for "server ready" and "thinking is off, OK" (twice:
once per teacher). If the run stops, attach this version's output and run again: finished jobs are skipped.

In [ ]:
# 0. Dead switch + settings: stops in seconds if anything is wrong, before the slow steps
import glob, os, json, time, re, subprocess, shutil, sys, random
from collections import Counter

TEACHERS = {
    "abliterated": ("huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF", "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf"),
    "original":    ("unsloth/Qwen3.8-27B-GGUF", "Qwen3.8-27B-UD-Q4_K_M.gguf"),
}
# (stage name, jobs file, teacher, kind). Stages run in this order; stages of one teacher are kept together.
STAGES = [
    ("rewrite",              "rewrite_jobs.jsonl",      "abliterated", "rewrite"),
    ("compare_abliterated",  "runB_compare_jobs.jsonl", "abliterated", "verdict"),
    ("compare_original",     "runB_compare_jobs.jsonl", "original",    "verdict"),
]
PARALLEL     = 8
CTX_PER_SLOT = 2048       # rewrite: ~700 tokens in + up to 800 out; Run B answers are shorter
PORT = 8080
INPUT, WORK = os.environ.get("KAGGLE_INPUT", "/kaggle/input"), os.environ.get("KAGGLE_WORK", "/kaggle/working")
FAKE = bool(os.environ.get("HATHER_FAKE"))      # only for testing this notebook against a fake server

problems, JOBS = [], {}
for stage, name, teacher, kind in STAGES:
    if name in JOBS: continue
    found = glob.glob(f"{INPUT}/**/{name}", recursive=True)
    if not found:
        problems.append(f"{name} not found under {INPUT} (Add Input -> your jobs dataset)"); continue
    try:
        jobs = [json.loads(l) for l in open(found[0], encoding="utf-8") if l.strip()]
    except Exception as e:
        problems.append(f"cannot read {found[0]}: {e}"); continue
    need = {"job_id", "messages", "schema", "sampling"} | ({"meta"} if kind == "verdict" else {"language", "scenario"})
    bad = [i for i, j in enumerate(jobs) if not need <= j.keys()]
    if bad: problems.append(f"{name}: {len(bad)} malformed jobs, first at line {bad[0]+1}")
    if len({j['job_id'] for j in jobs}) != len(jobs): problems.append(f"{name}: duplicate job_ids")
    JOBS[name] = jobs
try:
    open(f"{WORK}/.write_test", "w").write("ok"); os.remove(f"{WORK}/.write_test")
except Exception as e:
    problems.append(f"{WORK} not writable: {e}")
if not FAKE:
    gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
    if len(gpus) < 2: problems.append(f"need GPU T4 x2, found: {gpus or 'no GPU'}")
    if shutil.disk_usage("/tmp").free < 20e9: problems.append("less than 20 GB free in /tmp for the model")

if problems:
    print("STOPPING:\n - " + "\n - ".join(problems)); raise SystemExit(1)
for stage, name, teacher, kind in STAGES:
    print(f"OK  {stage:22} {len(JOBS[name]):5} jobs  teacher={teacher}")

In [ ]:
# 1. llama.cpp: reuse the llamacpp-t4 dataset if attached, otherwise build the newest release (~15-25 min)
#    Same build as the Run A notebook that worked on Oct 8: shell steps with full logs that stop on errors.
found = glob.glob(f"{INPUT}/**/llama-server", recursive=True)
BIN = f"{WORK}/llamacpp-bin"

def sh(cmd, log):
    # Run a shell command, keep the full output in a log, and show the real error if it fails
    with open(log, "w") as f:
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        lines = open(log, errors="replace").read().splitlines()
        hits = [i for i, l in enumerate(lines) if re.search(r"error|killed|undefined reference|signal 9", l, re.I)]
        shown = sorted({j for i in hits[:15] for j in range(i, min(i + 4, len(lines)))}) or range(max(0, len(lines) - 60), len(lines))
        print("\n".join(lines[j] for j in shown))
        raise RuntimeError(f"Failed (exit {rc}): {cmd[:60]}...  Full log: {log}")

if found:
    shutil.copytree(os.path.dirname(found[0]), BIN, dirs_exist_ok=True)
    print("using prebuilt llama.cpp from", found[0])
else:
    refs = subprocess.run("git ls-remote --tags https://github.com/ggml-org/llama.cpp",
                          shell=True, capture_output=True, text=True).stdout
    tag = "b%d" % max(int(n) for n in re.findall(r"refs/tags/b(\d+)$", refs, re.M))
    print("building llama.cpp", tag)
    shutil.rmtree("/tmp/llama.cpp", ignore_errors=True)
    sh(f"git clone --depth 1 --branch {tag} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp", f"{WORK}/clone.log")
    # GGML_CUDA_NO_VMM=ON: Kaggle's image has no libcuda.so for CMake to link, and VMM is optional
    sh("cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
       "-DGGML_CUDA_NO_VMM=ON -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF", f"{WORK}/cmake.log")
    print("compiling (silent for ~15-25 min)...", flush=True)
    sh("cmake --build /tmp/llama.cpp/build --config Release -j 4 --target llama-server", f"{WORK}/build.log")
    os.makedirs(BIN, exist_ok=True)
    shutil.copy("/tmp/llama.cpp/build/bin/llama-server", BIN)
    print("built. Add this notebook's output as a dataset named llamacpp-t4 to skip the build next time.")
os.chmod(f"{BIN}/llama-server", 0o755)
if not FAKE:
    print(subprocess.run(f"{BIN}/llama-server --version", shell=True, capture_output=True, text=True).stderr[-300:])

In [ ]:
# 2. Model and server helpers: download a teacher, start the server with thinking OFF, smoke test, stop
import urllib.request
MODEL_DIR = "/tmp/models"
server, current = None, None

XET_CACHE = os.path.expanduser("~/.cache/huggingface/xet")   # Hugging Face's download chunk cache

def download(teacher):
    repo, file = TEACHERS[teacher]
    shutil.rmtree(MODEL_DIR, ignore_errors=True)          # only one 16.5 GB model fits in /tmp at a time
    shutil.rmtree(XET_CACHE, ignore_errors=True)          # the first model's chunks would fill the disk too
    os.makedirs(MODEL_DIR, exist_ok=True)
    free = shutil.disk_usage("/tmp").free
    if not FAKE and free < 18e9:
        raise SystemExit(f"only {free/1e9:.1f} GB free in /tmp, need about 17 GB for {file}")
    t = time.time()
    if FAKE:
        path = f"{MODEL_DIR}/{file}"; open(path, "w").write("fake")
    else:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(repo, file, local_dir=MODEL_DIR)
        shutil.rmtree(XET_CACHE, ignore_errors=True)
    print(f"{teacher}: {path} {os.path.getsize(path)/1e9:.2f} GB in {time.time()-t:.0f}s")
    return path

def post(body, timeout=600):
    req = urllib.request.Request(f"http://localhost:{PORT}/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=timeout))

def body_for(job):
    return {"model": "teacher", "messages": job["messages"], **job["sampling"],
            "response_format": {"type": "json_schema", "json_schema": {"name": "out", "schema": job["schema"]}},
            "chat_template_kwargs": {"enable_thinking": False}}

def start_server(teacher, smoke_job):
    global server, current
    if current == teacher: return
    stop_server()
    model = download(teacher)
    cmd = [f"{BIN}/llama-server", "-m", model, "-ngl", "99", "--split-mode", "layer",
           "-c", str(CTX_PER_SLOT * PARALLEL), "--parallel", str(PARALLEL),
           "--jinja", "--chat-template-kwargs", '{"enable_thinking":false}', "--port", str(PORT)]
    log = f"{WORK}/server_{teacher}.log"
    server = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for i in range(180):                                   # up to 15 min to load
        try:
            if urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5).status == 200:
                print(f"server ready after {i*5}s"); break
        except Exception:
            pass
        if server.poll() is not None:
            print(open(log).read()[-3000:]); raise SystemExit("server crashed")
        time.sleep(5)
    else:
        raise SystemExit(f"server did not start; see {log}")
    for attempt in range(3):                               # the smoke test may hit a cold server
        try:
            msg = post(body_for(smoke_job))["choices"][0]["message"]; break
        except urllib.error.HTTPError as e:
            if e.code < 500 or attempt == 2:
                raise SystemExit(f"smoke test rejected: HTTP {e.code} {e.read()[:500]}")
            time.sleep(5)
    if msg.get("reasoning_content") or "<think>" in (msg.get("content") or ""):
        raise SystemExit("the model is still thinking: check --chat-template-kwargs")
    json.loads(msg["content"])
    print(f"thinking is off, OK ({teacher})\n", msg["content"][:300])
    current = teacher

def stop_server():
    global server, current
    if server is not None and server.poll() is None:
        server.terminate()
        try: server.wait(timeout=60)
        except subprocess.TimeoutExpired: server.kill(); server.wait()
    server, current = None, None

In [ ]:
# 3. The client: PARALLEL requests at a time, JSON schema enforced, 3 retries, every answer saved immediately
import asyncio, httpx

URL = f"http://localhost:{PORT}/v1/chat/completions"
ARABIC = re.compile(r"[ء-ي]")
PLACEHOLDER = re.compile(r"<(?:NAME|PHONE|IBAN|CARD|ID|PLATE|ADDRESS|EMAIL)>")

def arabic_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return sum(bool(ARABIC.match(c)) for c in letters) / max(1, len(letters))

def check_verdict(job, data):
    # Run B answer: verdict, confidence, signals, explanation (same rules as teacher_runB.ipynb)
    if data.get("verdict") not in ("scam", "legit"): raise ValueError(f"bad verdict {data.get('verdict')!r}")
    conf = data.get("confidence")
    if not isinstance(conf, int) or not 0 <= conf <= 100: raise ValueError(f"bad confidence {conf!r}")
    expl = (data.get("explanation") or "").strip()
    if not expl: raise ValueError("empty explanation")
    answer = {"verdict": data["verdict"], "confidence": conf,
              "signals": [s.strip() for s in data.get("signals", []) if s.strip()][:5], "explanation": expl}
    problems = []
    if len(expl.split()) > 40: problems.append("long_explanation")
    text = job["messages"][-1]["content"].split("[TEXT]", 1)[-1].strip()
    if (arabic_ratio(text) >= 0.5) != (arabic_ratio(expl) >= 0.5): problems.append("wrong_language")
    return answer, problems

def check_rewrite(job, data):
    # Rewrite answer: one message {sender, text}
    items = data.get("messages") or []
    if not items: raise ValueError("no message returned")
    it = items[0]
    text, sender = (it.get("text") or "").strip(), (it.get("sender") or "").strip()
    if len(text) < 5: raise ValueError("empty message")
    original = job["messages"][-1]["content"].split("\n\nOriginal sender:")[0].split(":\n", 1)[-1].strip()
    problems = []
    r = arabic_ratio(text)
    if (job["language"] == "ar" and r < 0.5) or (job["language"] == "en" and r > 0.2): problems.append("wrong_language")
    if set(PLACEHOLDER.findall(original)) - set(PLACEHOLDER.findall(text)): problems.append("placeholder_lost")
    if text == original: problems.append("unchanged")
    return {"sender": sender, "text": text}, problems

CHECKS = {"verdict": check_verdict, "rewrite": check_rewrite}

def out_path(stage): return f"{WORK}/prep_{stage}_out.jsonl"

def load_done(path):
    done = set()
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        if r.get("ok"): done.add(r["job_id"])
    return done

def carry_over(stage):
    # Copy finished answers from an attached earlier version's output, once
    path = out_path(stage)
    previous = glob.glob(f"{INPUT}/**/{os.path.basename(path)}", recursive=True)
    if previous and not os.path.exists(path):
        with open(path, "w", encoding="utf-8") as f:
            for p in previous:
                f.writelines(l for l in open(p, encoding="utf-8") if json.loads(l).get("ok"))
    return load_done(path) if os.path.exists(path) else set()

async def run(stage, jobs, kind, parallel=PARALLEL, retries=3):
    path = out_path(stage)
    done = carry_over(stage)
    todo = [j for j in jobs if j["job_id"] not in done]
    print(f"[{stage}] {len(done)} already done, {len(todo)} to go", flush=True)
    if not todo: return
    check = CHECKS[kind]
    sem, lock = asyncio.Semaphore(parallel), asyncio.Lock()
    stats = {"n": 0, "fail": 0, "t0": time.time()}
    out = open(path, "a", encoding="utf-8")

    async def one(client, job):
        async with sem:
            rec = {"job_id": job["job_id"], "ok": False}
            for attempt in range(retries):
                t = time.time()
                try:
                    r = await client.post(URL, json=body_for(job), timeout=900)   # as in Run A
                    if r.status_code != 200:
                        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
                    msg = r.json()["choices"][0]["message"]
                    content = re.sub(r"<think>.*?</think>", "", msg.get("content") or "", flags=re.S).strip()
                    answer, problems = check(job, json.loads(content))
                    rec = {"job_id": job["job_id"], "ok": True, "answer": answer, "problems": problems,
                           "seconds": round(time.time() - t, 1), "thinking": bool(msg.get("reasoning_content"))}
                    break
                except Exception as e:
                    rec["error"] = f"{type(e).__name__}: {str(e)[:300]}"
            async with lock:
                out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
                stats["n"] += 1; stats["fail"] += not rec["ok"]
                if stats["n"] % 50 == 0 or stats["n"] == len(todo):
                    el = time.time() - stats["t0"]; rate = stats["n"] / el
                    print(f"[{stage}] {stats['n']}/{len(todo)}  failed={stats['fail']}  {rate*60:.1f} jobs/min  "
                          f"ETA {(len(todo)-stats['n'])/rate/60:.0f} min", flush=True)

    async with httpx.AsyncClient(limits=httpx.Limits(max_connections=parallel)) as client:
        await asyncio.gather(*(one(client, j) for j in todo))
    out.close()

def results(stage, jobs):
    # Latest answer per job_id (a success wins over earlier failures)
    by_id = {j["job_id"]: j for j in jobs}
    latest = {}
    for line in open(out_path(stage), encoding="utf-8"):
        r = json.loads(line)
        if r["job_id"] in by_id and (r.get("ok") or r["job_id"] not in latest):
            latest[r["job_id"]] = r
    return latest

print("client ready")

In [ ]:
# 4. Run the stages (one model download per teacher). Each stage resumes on its own.
for stage, name, teacher, kind in STAGES:
    jobs = JOBS[name]
    if all(j["job_id"] in carry_over(stage) for j in jobs):  # no model download for a finished stage
        print(f"[{stage}] all {len(jobs)} jobs already done, skipping"); continue
    start_server(teacher, jobs[0])
    t = time.time()
    await run(stage, jobs, kind)
    for retry in range(2):                                 # redo failures twice before moving on
        failed = [j for j in jobs if not results(stage, jobs).get(j["job_id"], {}).get("ok")]
        if not failed: break
        print(f"[{stage}] retrying {len(failed)} failed jobs")
        await run(stage, failed, kind)
    print(f"[{stage}] finished in {(time.time()-t)/60:.0f} min\n")
stop_server()

In [ ]:
# 5. Outputs: rewrite messages, comparison labels, and the summary (download all into hather/data/out/kaggle_prep/)
summary = ["# Teacher prep run", ""]

# 5a. rewrites -> pool-format records for data/build_mix.py --rewrites
stage, name = "rewrite", "rewrite_jobs.jsonl"
jobs = {j["job_id"]: j for j in JOBS[name]}
rows = results(stage, JOBS[name])
recs, probs = [], Counter()
for jid, r in rows.items():
    if not r.get("ok"): continue
    for p in r["problems"]: probs[p] += 1
    j = jobs[jid]
    recs.append({"id": jid, "source": j.get("source", "sms"), "language": j["language"],
                 "sender": r["answer"]["sender"], "subject": "", "text": r["answer"]["text"],
                 "human_label": "scam", "origin": "teacher_rewrite", "eml_path": None,
                 "parent_id": None, "scenario": j["scenario"], "rewrite_of": j.get("rewrite_of"),
                 "trick": j.get("trick"), "sender_style": j.get("sender_style"), "problems": r["problems"]})
with open(f"{WORK}/rewrite_messages.jsonl", "w", encoding="utf-8") as f:
    for x in recs: f.write(json.dumps(x, ensure_ascii=False) + "\n")
n_fail = sum(1 for r in rows.values() if not r.get("ok")) + len(set(jobs) - set(rows))
summary += ["## Rewrites", "", f"{len(recs)} of {len(jobs)} jobs answered ({n_fail} failed) -> `rewrite_messages.jsonl`",
            f"by language: {dict(Counter(x['language'] for x in recs))}",
            f"problems (kept, build_mix.py drops wrong-language ones): {dict(probs)}", ""]
print("\n".join(summary[-5:]))
for x in random.sample(recs, min(4, len(recs))):
    print(f"\n[{x['language']} | trick {x['trick']} | {x['sender_style']} | {x['sender']}]\n{x['text'][:400]}")

# 5b. comparison -> labels files (same format as teacher_runB.ipynb) + agreement table
summary += ["## Teacher comparison (100 English test SMS)", "",
            "| Teacher | Answered | Agreement (ALL) | Scams caught | Legit kept | Problems |",
            "| --- | ---: | ---: | ---: | ---: | --- |"]
scores = {}
for stage, name, teacher, kind in STAGES:
    if kind != "verdict": continue
    jobs = JOBS[name]; rows = results(stage, jobs)
    with open(f"{WORK}/runB_compare_{teacher}_labels.jsonl", "w", encoding="utf-8") as f:
        for j in jobs:
            r = rows.get(j["job_id"], {"ok": False, "error": "not run"})
            f.write(json.dumps({"id": j["job_id"], "ok": r.get("ok", False), "teacher": teacher,
                                "answer": r.get("answer"), "problems": r.get("problems", []),
                                "error": r.get("error"), "meta": j["meta"]}, ensure_ascii=False) + "\n")
    ok = [(j, rows[j["job_id"]]) for j in jobs if rows.get(j["job_id"], {}).get("ok")]
    agree = sum(r["answer"]["verdict"] == j["meta"]["label"] for j, r in ok)
    scam = [(j, r) for j, r in ok if j["meta"]["label"] == "scam"]
    legit = [(j, r) for j, r in ok if j["meta"]["label"] == "legit"]
    rec = sum(r["answer"]["verdict"] == "scam" for j, r in scam) / max(1, len(scam))
    keep = sum(r["answer"]["verdict"] == "legit" for j, r in legit) / max(1, len(legit))
    all_ = agree / len(jobs)                               # unanswered jobs count as wrong
    scores[teacher] = (all_, rec, keep)
    pr = dict(Counter(p for j, r in ok for p in r["problems"]))
    summary.append(f"| {teacher} | {len(ok)}/{len(jobs)} | {all_:.1%} | {rec:.1%} | {keep:.1%} | {pr or '-'} |")
if scores:
    best = max(scores, key=lambda t: (scores[t][0], scores[t][1]))
    tie = [t for t in scores if scores[t][0] == scores[best][0]]
    same = [t for t in tie if scores[t][:2] == scores[best][:2]]
    if len(same) > 1:                                      # same agreement and same scams caught
        best = "abliterated" if "abliterated" in same else best
        note = "(a full tie: keeping the abliterated teacher, the plan's default)"
    elif len(tie) > 1:
        note = f"(tie on agreement; {best} catches more scams)"
    else:
        note = ""
    summary += ["", f"**Winner for Run B: `{best}`** {note}".strip(),
                "Set `TEACHER = \"%s\"` in teacher_runB.ipynb." % best]
open(f"{WORK}/prep_summary.md", "w", encoding="utf-8").write("\n".join(summary) + "\n")
print("\n" + "\n".join(summary[summary.index("## Teacher comparison (100 English test SMS)"):]))
print("\nfiles:", sorted(os.path.basename(p) for p in glob.glob(f"{WORK}/*.jsonl") + glob.glob(f"{WORK}/*.md")))

**When it finishes:** from the Output tab, download `rewrite_messages.jsonl`, `runB_compare_abliterated_labels.jsonl`,
`runB_compare_original_labels.jsonl` and `prep_summary.md` into `hather/data/out/kaggle_prep/`. Then on your laptop:

```
python data/build_mix.py --rewrites data/out/kaggle_prep/rewrite_messages.jsonl
python data/make_runB_jobs.py data/out/runB_pool_v2.jsonl -o data/out/runB_jobs_v2.jsonl
```

and run `teacher_runB.ipynb` with `JOBS_NAME = "runB_jobs_v2.jsonl"` and the winning `TEACHER`.